# 🎙️ VoiceCloner — Viterbox TTS trên Colab

Repo: https://github.com/kanazawahere/VoiceCloner

**Tính năng:**
- ✅ Python 3.12 compatible (numpy>=1.26)
- ✅ HuggingFace cache lưu vào Google Drive
- ✅ Zero-shot voice cloning
- ✅ Xử lý văn bản dài

In [ ]:
# ✅ BƯỚC 1: Trỏ HuggingFace cache về Drive
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_HF_CACHE = "/content/drive/MyDrive/viterbox_hf_cache"
os.makedirs(DRIVE_HF_CACHE, exist_ok=True)

os.environ['HF_HOME'] = DRIVE_HF_CACHE
os.environ['HUGGINGFACE_HUB_CACHE'] = DRIVE_HF_CACHE

print(f"✅ HF cache: {DRIVE_HF_CACHE}")

In [ ]:
# ✅ BƯỚC 2: Cài uv
!curl -LsSf https://astral.sh/uv/install.sh | sh

import os
current_path = os.environ.get('PATH', '')
new_path = f"/root/.local/bin:{current_path}"
os.environ['PATH'] = new_path
%env PATH={new_path}
!uv --version

In [ ]:
# ✅ BƯỚC 3: Clone repo
import shutil
if os.path.exists('/content/VoiceCloner'):
    shutil.rmtree('/content/VoiceCloner')

!git clone https://github.com/kanazawahere/VoiceCloner.git \
    /content/VoiceCloner --depth=1 --quiet
%cd /content/VoiceCloner
print('✅ Clone xong')

In [ ]:
# ✅ BƯỚC 4: Cài packages
import subprocess
cuda_ver = subprocess.getoutput(
    r"nvcc --version | grep 'release' | sed 's/.*release //' | sed 's/,.*//' | sed 's/\.//'"
).strip()
if not cuda_ver.isdigit():
    cuda_ver = "121"
torch_index = f"https://download.pytorch.org/whl/cu{cuda_ver}"
print(f"CUDA: cu{cuda_ver}")

# Cài torch GPU TRƯỚC
!uv pip install --system torch torchvision torchaudio \
    --index-url {torch_index} --quiet

# Cài requirements (bỏ qua torch/torchaudio)
!uv pip install --system -r requirements.txt --quiet
!uv pip install --system -e . --quiet

print("✅ Cài xong")
print("⚠️ QUAN TRỌNG: Chạy cell tiếp theo để restart runtime!")

In [ ]:
# ✅ BƯỚC 4.5: Restart runtime để load lại numpy
import os
os.kill(os.getpid(), 9)

In [ ]:
# ✅ BƯỚC 5: Kiểm tra môi trường sau restart
import os
print(f"HF_HOME: {os.environ.get('HF_HOME', 'NOT SET')}")
print(f"Working dir: {os.getcwd()}")

# Set lại env vars nếu cần
if 'HF_HOME' not in os.environ:
    DRIVE_HF_CACHE = "/content/drive/MyDrive/viterbox_hf_cache"
    os.environ['HF_HOME'] = DRIVE_HF_CACHE
    os.environ['HUGGINGFACE_HUB_CACHE'] = DRIVE_HF_CACHE
    print(f"✅ Set HF_HOME: {DRIVE_HF_CACHE}")

# Chuyển về thư mục VoiceCloner
if not os.path.exists('/content/VoiceCloner'):
    print("⚠️ Thư mục VoiceCloner không tồn tại. Chạy lại từ BƯỚC 3!")
else:
    %cd /content/VoiceCloner
    print("✅ Sẵn sàng")

In [ ]:
# ✅ BƯỚC 6: Load model
import torch
from viterbox import Viterbox

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

tts = Viterbox.from_pretrained(device)
print("✅ Model load xong!")

In [ ]:
# ✅ BƯỚC 7: Test TTS cơ bản
from IPython.display import Audio, display

text = "Xin chào, tôi là Viterbox, hệ thống chuyển văn bản thành giọng nói tiếng Việt."

audio = tts.generate(text=text, language="vi")
tts.save_audio(audio, "/content/output_basic.wav")
display(Audio("/content/output_basic.wav"))
print("✅ Done!")

---
## 🎤 Voice Cloning (Nhân bản giọng)

Upload file audio giọng mẫu (3-10 giây, sạch không nhiễu)

In [ ]:
# ✅ BƯỚC 8: Upload giọng mẫu
from google.colab import files

print("📤 Upload file WAV giọng mẫu (3-10 giây, sạch không nhiễu):")
uploaded = files.upload()
ref_audio = list(uploaded.keys())[0]
print(f"✅ Đã upload: {ref_audio}")

In [ ]:
# ✅ BƯỚC 9: Clone giọng nói
from IPython.display import Audio, display

text_clone = """Đây là bài kiểm tra nhân bản giọng nói với Viterbox.
Công nghệ zero-shot voice cloning cho phép mô phỏng giọng nói chỉ với vài giây audio mẫu.
Kết quả sẽ giống với giọng gốc mà bạn đã cung cấp."""

audio_cloned = tts.generate(
    text=text_clone,
    language="vi",
    audio_prompt=ref_audio,
    sentence_pause_ms=600
)

tts.save_audio(audio_cloned, "/content/output_cloned.wav")
display(Audio("/content/output_cloned.wav"))
print("✅ Voice cloning xong!")

---
## 📝 Văn bản dài

Tự động chia câu và ghép audio mượt mà

In [ ]:
# ✅ BƯỚC 10: Xử lý văn bản dài
from IPython.display import Audio, display

long_text = """Việt Nam là một quốc gia nằm ở phía đông bán đảo Đông Dương.
Đất nước có hình chữ S với chiều dài hơn 1600 km.
Thủ đô Hà Nội là trung tâm văn hóa của cả nước.
Thành phố Hồ Chí Minh là trung tâm kinh tế lớn nhất.
Việt Nam có 54 dân tộc anh em cùng chung sống hòa bình."""

audio_long = tts.generate(
    text=long_text,
    language="vi",
    sentence_pause_ms=500  # nghỉ 0.5s giữa các câu
)

tts.save_audio(audio_long, "/content/output_long.wav")
display(Audio("/content/output_long.wav"))
print("✅ Văn bản dài xong!")

---
## 💾 Lưu output vào Drive

In [ ]:
# ✅ BƯỚC 11: Lưu tất cả output vào Drive
import shutil
import os

OUTPUT_DIR = "/content/drive/MyDrive/viterbox_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

files_to_save = [
    "output_basic.wav",
    "output_cloned.wav",
    "output_long.wav"
]

for fname in files_to_save:
    src = f"/content/{fname}"
    if os.path.exists(src):
        shutil.copy(src, f"{OUTPUT_DIR}/{fname}")
        print(f"✅ Đã lưu: {fname}")
    else:
        print(f"⏭️  Bỏ qua (chưa tạo): {fname}")

print(f"\n📁 Output folder: {OUTPUT_DIR}")

---
## 📋 Ghi chú

### Tham số `generate()`
```python
tts.generate(
    text="...",              # văn bản cần đọc
    language="vi",           # "vi" hoặc "en"
    audio_prompt="ref.wav", # None = giọng mặc định
    sentence_pause_ms=500,   # thời gian nghỉ giữa các câu (ms)
)
```

### Tips
- Giọng mẫu tốt nhất: 3-10 giây, sạch, không nhiễu
- Văn bản dài: tự động chia câu, tối đa ~500 từ
- Model cache lưu trong Drive, lần sau load nhanh hơn